<a href="https://colab.research.google.com/github/weagan/ARC-AGI-3/blob/main/arc_agi3_multigame_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARC-AGI-3 Multi-Game Goal Discovery & Execution Orchestrator
This notebook provides a production-grade implementation of an advanced **Multi-Game Duck Harness** [cite: 1]. Guided by the **Official ARC-AGI-3 System Prompt**, it ingests two entirely distinct game types (Agent Movement vs. Global Orchestration), prompts an LLM to discover their unique rulesets, and executes targeted structural code to solve them inside a unified scope sandbox [cite: 1].

### 1. Secure Multi-Platform Credential Handler
Dynamically loads the execution tokens safely across Colab and Kaggle platforms [cite: 1].

In [1]:
import os

def load_api_key(secret_name="GEMINI_API_KEY"):
    """Loads a secret securely from Google Colab or Kaggle secret managers [cite: 1]."""
    try:
        from google.colab import userdata
        return userdata.get(secret_name)
    except ImportError:
        pass

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(secret_name)
    except ImportError:
        pass

    return os.environ.get(secret_name, "")

API_KEY = load_api_key()
print("Secret loaded successfully (masked for privacy) [cite: 1].")

Secret loaded successfully (masked for privacy) [cite: 1].


### 2. Multi-Game Environment Suite Definition
Defines two distinctly different game matrices:
1. **Game A (Agent Movement / Maze):** Controls an individual entity via 4-directional steps [cite: 1].
2. **Game B (Global Grid Orchestration / Tiling):** Manipulates coordinates directly using a global confirmation matrix [cite: 1].

In [2]:
import numpy as np

class AgentMovementGame:
    """Game Type A: Entity navigation via step primitives [cite: 1]."""
    def __init__(self):
        self.control_layout = "AGENT_MOVEMENT_KEYS"
        self.grid = np.array([
            [1, 1, 1, 1, 1],
            [1, 2, 0, 0, 1],
            [1, 1, 1, 0, 1],
            [1, 3, 0, 0, 1],
            [1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        if self.grid[r, c] != 1:
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("[Game Engine A] Success: Goal achieved!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2
        return self.get_state()

class GlobalOrchestrationGame:
    """Game Type B: Pattern completion via global frame mutations [cite: 1]."""
    def __init__(self):
        self.control_layout = "GLOBAL_MATRIX_EDIT_KEYS"
        # Incomplete symmetric pattern. Object is to match the left 4 with the right 4.
        self.grid = np.array([
            [4, 0, 0, 0],
            [4, 0, 0, 0],
            [4, 0, 0, 0],
            [4, 0, 0, 4]  # Missing elements in center columns
        ])

    def get_state(self):
        return self.grid.copy()

    def update_cell(self, row, col, value):
        if 0 <= row < 4 and 0 <= col < 4:
            self.grid[row, col] = value

    def click_confirm(self):
        # Check if symmetric matrix condition is reached
        if np.array_equal(self.grid[:, 0], self.grid[:, 3]):
            print("[Game Engine B] Success: Global Symmetry Criteria Validated!")
        else:
            print("[Game Engine B] Failed: Pattern does not meet criteria.")
        return self.get_state()

### 3. Orchestration Harness & Dual-Phase Discovery Prompter
Implements the exact **Official ARC-AGI-3 System Prompt** and integrates a secure unified execution scope sandbox [cite: 1].

In [3]:
import sys
from io import StringIO

class MultiGameDuckHarness:
    def __init__(self):
        self.official_system_prompt = (
            "You are playing a game. Your goal is to win. Reply with the exact action you want to take. "
            "The final action in your reply will be executed next turn. Your entire reply will be carried to the next turn."
        )

    def serialize_grid(self, grid):
        return "\n".join([" ".join(map(str, row)) for row in grid])

    def assemble_discovery_prompt(self, grid, control_type):
        grid_str = self.serialize_grid(grid)
        return f"""=== ARC-AGI-3 CORE SYSTEM ENVIRONMENT ===
{self.official_system_prompt}

=== CURRENT GAME WORKING BLANK INTERFACE ===
Grid Array:
{grid_str}

Available Control Layer Protocol Signature: {control_type}

=== TASKS ===
1. Analyze the grid layout and deduce the goal of the game using Core Knowledge priors.
2. Generate valid executable Python code using the available primitives for this layout.
   - For AGENT_MOVEMENT_KEYS, you have access to: `env.step('UP'/'DOWN'/'LEFT'/'RIGHT')` [cite: 1]
   - For GLOBAL_MATRIX_EDIT_KEYS, you have access to: `env.update_cell(r, c, val)` and `env.click_confirm()` [cite: 1]

Return ONLY the executable Python script block enclosed inside a markdown sequence.
"""

    def execute_code_repl(self, code_str, env):
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output

        # Combined global and local execution scope to prevent scoping NameErrors [cite: 1].
        execution_scope = {"env": env, "harness": self, "__builtins__": __builtins__}
        try:
            exec(code_str, execution_scope, execution_scope)
            error = None
        except Exception as e:
            error = str(e)

        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

### 4. Running the Multi-Game Processing Loop
Orchestrates cross-game processing automatically. It builds custom prompts per control signature, executes code solutions, and displays verified results cleanly [cite: 1].

In [4]:
from google import genai
import re

client = genai.Client(api_key=API_KEY)
harness = MultiGameDuckHarness()

# Instantiate our distinct game configurations
games = [
    ("Game Type A (Agent Step Maze)", AgentMovementGame()),
    ("Game Type B (Global Matrix Orchestration)", GlobalOrchestrationGame())
]

for game_title, env in games:
    print(f"\n=======================================================")
    print(f"STARTING ORCHESTRATION PIPELINE FOR: {game_title}")
    print(f"=======================================================")

    # 1. Compile prompt context dynamically mapped to active constraints [cite: 1]
    prompt = harness.assemble_discovery_prompt(env.get_state(), env.control_layout)

    if not API_KEY:
        print("[Notice] API key absent. Emulating the specific layout code-as-action script block fallback...")
        if env.control_layout == "AGENT_MOVEMENT_KEYS":
            script_block = """
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
"""
        else:
            script_block = """
env.update_cell(0, 3, 4)
env.update_cell(1, 3, 4)
env.update_cell(2, 3, 4)
env.click_confirm()
"""
    else:
        print("--- Communicating with Gemini Model API for Goal Induction... ---")
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        blocks = re.findall(r'```python\s*(.*?)\s*```', response.text, re.DOTALL)
        script_block = blocks[0] if blocks else response.text

    print("\n--- Executable Python Script Decoded via Harness Factory ---")
    print(script_block.strip())

    print("\n--- Executing Action Block via REPL Sandbox ---")
    output, err = harness.execute_code_repl(script_block, env)

    if err:
        print(f"Runtime Evaluation Error: {err} [cite: 1]")
    else:
        print(f"Harness Log Out:\n{output if output else 'Script evaluated successfully with zero boundary exceptions. [cite: 1]'}")

    print("\n--- Verified Final State Array Layout ---")
    print(harness.serialize_grid(env.get_state()))


STARTING ORCHESTRATION PIPELINE FOR: Game Type A (Agent Step Maze)
--- Communicating with Gemini Model API for Goal Induction... ---

--- Executable Python Script Decoded via Harness Factory ---
# Deduced Goal: Move the agent (represented by '2') to the target position (represented by '3').
# Agent starts at grid position (1,1). Target is at grid position (3,1).

# Path to target (3,1):
# 1. From (1,1), move RIGHT to (1,2)
# 2. From (1,2), move RIGHT to (1,3)
# 3. From (1,3), move DOWN to (2,3)
# 4. From (2,3), move DOWN to (3,3)
# 5. From (3,3), move LEFT to (3,2)
# 6. From (3,2), move LEFT to (3,1) - Target Reached

env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
env.step('LEFT')

--- Executing Action Block via REPL Sandbox ---
Harness Log Out:
[Game Engine A] Success: Goal achieved!


--- Verified Final State Array Layout ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 2 0 0 1
1 1 1 1 1

STARTING ORCHESTRATION PIPELINE FOR: Game Type B (Global Matrix Orch